# Building X and y for speech separation

**Task:** the model hears one recording with two people talking (**X**) and
has to give back each person's voice separately (**y**).

Every conversation in the dataset was recorded with one track per speaker,
so we can create both:

| | What it is | Shape (for a window) |
|---|---|---|
| **X** | both speakers mixed into one track: `(a + b) / 2` | `(samples,)` |
| **y** | the two original tracks, `[a, b]` | `(2, samples)` |

**No random split:** `raw_data/train`, `val` and `test` were already split
by speaker (`scripts/organize_splits.sh`), so a voice the model is tested
on never appears in training. Reading a folder *is* the split.

**Steps**
1. Setup
2. Fuse the two tracks of every conversation into `fused.flac`
3. Build `X` (fused files) and `y` (pairs of speaker files) per split
4. Load a short window of audio for training
5. Check one example by looking and listening

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from tqdm.auto import tqdm

RAW_DIR = Path("../raw_data")  # raw_data/<split>/<conversation>/
SAMPLE_RATE = 16_000           # all audio is 16 kHz mono
BLOCK_SECONDS = 60             # fuse 60 s at a time to keep memory low

## 2. Fuse the speaker tracks

For each conversation we write `fused.flac` next to the two speaker
files. The mix is `(a + b) / 2`, the same method as `fuse.ipynb`.
Conversations that are already fused are skipped, so this is fast the
second time.

In [ ]:
def fuse_conversation(conv_dir):
    """Mix speaker_a and speaker_b into conv_dir/fused.flac.

    Parameters
    ----------
    conv_dir : Path
        Folder that contains speaker_a.flac and speaker_b.flac.

    Returns
    -------
    Path
        Path of the fused file.
    """
    out_path = conv_dir / "fused.flac"
    if out_path.exists():
        return out_path

    path_a = conv_dir / "speaker_a.flac"
    path_b = conv_dir / "speaker_b.flac"
    with sf.SoundFile(path_a) as file_a, sf.SoundFile(path_b) as file_b:
        length = min(len(file_a), len(file_b))
        block = BLOCK_SECONDS * file_a.samplerate
        with sf.SoundFile(out_path, "w", samplerate=file_a.samplerate,
                          channels=1, subtype="PCM_24") as out:
            for start in range(0, length, block):
                n_frames = min(block, length - start)
                a = file_a.read(n_frames, dtype="float32", always_2d=True)
                b = file_b.read(n_frames, dtype="float32", always_2d=True)
                out.write((a.mean(axis=1) + b.mean(axis=1)) / 2)
    return out_path

## 3. Build X and y for each split

`X` is a list of fused files and `y` the matching list of
`(speaker_a, speaker_b)` files. Item *i* of `X` and item *i* of `y` always
belong to the same conversation. Folders are sorted, so the order is the
same on every computer.

In [ ]:
def list_conversations(split):
    """Return the conversation folders of one split, sorted by name."""
    return sorted(p for p in (RAW_DIR / split).iterdir() if p.is_dir())


def get_xy(split):
    """Return X (fused files) and y (speaker file pairs) for one split."""
    conversations = list_conversations(split)
    X = [fuse_conversation(conv)
         for conv in tqdm(conversations, desc=f"fusing {split}")]
    y = [(conv / "speaker_a.flac", conv / "speaker_b.flac")
         for conv in conversations]
    return X, y


X_train, y_train = get_xy("train")
X_val, y_val = get_xy("val")
X_test, y_test = get_xy("test")

print(f"train: {len(X_train)}  val: {len(X_val)}  test: {len(X_test)}")

## 4. Load a window of audio

Conversations can be hours long, so we never load a whole file. This
reads only the requested seconds from the three files. `y` is divided by
2 so that `x == y[0] + y[1]`, matching how the fused file was made.

In [ ]:
def load_window(x_path, y_paths, start_sec, duration_sec):
    """Load one window of X and y.

    Returns
    -------
    x : np.ndarray, shape (samples,)
        The fused (mixed) audio.
    y : np.ndarray, shape (2, samples)
        Speaker A and speaker B, scaled to match x.
    """
    start = int(start_sec * SAMPLE_RATE)
    n_frames = int(duration_sec * SAMPLE_RATE)
    x, _ = sf.read(x_path, start=start, frames=n_frames, dtype="float32")
    a, _ = sf.read(y_paths[0], start=start, frames=n_frames, dtype="float32")
    b, _ = sf.read(y_paths[1], start=start, frames=n_frames, dtype="float32")
    return x, np.stack([a, b]) / 2

## 5. Check one example

What to expect:
- all three files are 16 kHz, mono and the same length
- `x` equals `y[0] + y[1]`
- wherever speaker A or B has sound, the fused track has it too
- the fused clip has both voices, the others one voice each

In [ ]:
EXAMPLE = 0          # which training conversation to check
START_SEC = 60       # where the window starts
DURATION_SEC = 10    # window length

x_path, y_paths = X_train[EXAMPLE], y_train[EXAMPLE]
print("Conversation:", x_path.parent.name)

for name, path in [("fused", x_path), ("speaker_a", y_paths[0]),
                   ("speaker_b", y_paths[1])]:
    info = sf.info(path)
    print(f"{name:<10} {info.samplerate} Hz  {info.channels} channel  "
          f"{info.duration / 60:.1f} min")

x, y = load_window(x_path, y_paths, START_SEC, DURATION_SEC)
print("\nx:", x.shape, " y:", y.shape)
print("x == y[0] + y[1]:", np.allclose(x, y[0] + y[1], atol=1e-4))

In [ ]:
signals = [(x, "X: fused"), (y[0], "y: speaker A"), (y[1], "y: speaker B")]
seconds = np.arange(len(x)) / SAMPLE_RATE

fig, axes = plt.subplots(3, 1, figsize=(12, 5), sharex=True)
for ax, (signal, title) in zip(axes, signals):
    ax.plot(seconds, signal, linewidth=0.5)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("amplitude")
axes[-1].set_xlabel(f"time (seconds from {START_SEC} s into the conversation)")
fig.suptitle(f"{x_path.parent.name}: fused input X and the two targets y")
plt.tight_layout()
plt.show()

for signal, title in signals:
    print(title)
    display(Audio(signal, rate=SAMPLE_RATE))